In [1]:
import textgrid
import math
import cv2
import numpy as np
import os
from tqdm import tqdm

In [2]:
# ########## MAIN MAPPING

# phoneme2viseme = {
#     'AA':0,
#     'AE':1,
#     'AH':3,
#     'AO':5,
#     'AW':6,
#     'AY':2,
#     'AX':0,
#     'B':18,
#     'CH':16,
#     'D':16,
#     'DH':16,
#     'EH':3,
#     'ER':3,
#     'EY':1,
#     'F':19,
#     'G':3,
#     'HH':3,
#     'IH':8,
#     'IY':7,
#     'JH':16,
#     'K':15,
#     'L':17,
#     'M':18,
#     'N':17,
#     'NG':7,
#     'OW':10,
#     'OY':10,
#     'P':18,
#     'R':17,
#     'S':15,
#     'SH':15,
#     'T':16,
#     'TH':16,
#     'UH':13,
#     'UW':13,
#     'UX':13,
#     'V':19,
#     'W':13,
#     'Y':7,
#     'Z':16,
#     'ZH':16
# }

In [3]:
phoneme2viseme = {
    'AA':2,
    'AE':7, 
    'AH':16,
    'AO':8,
    'AW':8,
    'AY':4,
    'AX':2,
    'B':1, 
    'CH':14,
    'D':13,
    'DH':14,
    'EH':7,
    'ER':7,
    'EY':2,
    'F':15,
    'G':11,
    'HH':11,
    'IH':1,
    'IY':4,
    'JH':14,
    'K':11,
    'L':13,
    'M':1,
    'N':13,
    'NG':11,
    'OW':8,
    'OY':8,
    'P':1,
    'R':13,
    'S':14,
    'SH':10,
    'T':14,
    'TH':14,
    'UH':17,
    'UW':17,
    'UX':17,
    'V':15,
    'W':17,
    'Y':4,
    'Z':12,
    'ZH':12
}

In [11]:
# set paths
ALIGNER_ROOT = '/mnt/users_scratch/astitva/WORKSPACE/Fedora-DGX-Codebase/GENERATION/Audio2Presets/ForcedAligner/'
ASSETS_ROOT = '/mnt/users_scratch/astitva/WORKSPACE/Fedora-DGX-Codebase/GENERATION/PresetGeneration/OUTPUT/DRAWINGS/'
VIDEO_SAVE_DIR = './VIDEO_ANIM/'

In [ ]:
filename = 'babyshark'
tg_path = f'{ALIGNER_ROOT}/outputs/{filename}.TextGrid'
tg = textgrid.TextGrid.fromFile(tg_path)
tg

TextGrid(None, [IntervalTier(words, [Interval(0.0, 0.88, None), Interval(0.88, 1.11, baby), Interval(1.11, 1.91, None), Interval(1.91, 2.04, shark), Interval(2.04, 2.41, doo), Interval(2.41, 2.58, None), Interval(2.58, 2.76, doo), Interval(2.76, 3.04, doo), Interval(3.04, 3.51, None), Interval(3.51, 4.01, doo), Interval(4.01, 4.16, None), Interval(4.16, 4.28, doo), Interval(4.28, 4.45, doo), Interval(4.45, 4.55, None), Interval(4.55, 4.99, baby), Interval(4.99, 5.09, None), Interval(5.09, 5.35, shark), Interval(5.35, 5.46, doo), Interval(5.46, 5.57, None), Interval(5.57, 5.81, doo), Interval(5.81, 5.85, None), Interval(5.85, 5.96, doo), Interval(5.96, 6.1, doo), Interval(6.1, 6.25, None), Interval(6.25, 6.36, doo), Interval(6.36, 6.54, doo), Interval(6.54, 6.64, None), Interval(6.64, 7.09, baby), Interval(7.09, 7.18, None), Interval(7.18, 7.43, shark), Interval(7.43, 7.53, doo), Interval(7.53, 7.66, None), Interval(7.66, 7.92, doo), Interval(7.92, 7.96, None), Interval(7.96, 8.07, doo)

In [6]:
words = tg[0]
phonemes = tg[1]

words_phonemes=[]
phoneme_idx = 0
for w in words:
    current_set = []
    time = w.duration()
    start = 0
    while(time!=start):
        ph = phonemes[phoneme_idx]
        start += ph.duration()
        current_set.append(ph)
        phoneme_idx += 1
    words_phonemes.append(current_set)

In [14]:
character_id = '07aedcb335a04981a016c0c7efed77ba'
os.makedirs(f'{VIDEO_SAVE_DIR}/{character_id}', exist_ok=True)
face_only = True

suffix = ''
if face_only:
    suffix = '_face'

fps=240

video=cv2.VideoWriter(f'tmp.mp4',cv2.VideoWriter_fourcc(*'DIVX'),fps,(1024,1024))
buffer = np.ones((1024,1024,3)).astype('uint8')*255
for i in tqdm(range(len(words))):
    total_duration = words[i].duration()
    word_frame_count = int(fps*total_duration)
    cumulative_frame_count = 0
    for p in words_phonemes[i]:
        # frame_count = int((fps*p.duration())/total_duration))
        frame_count = math.ceil(fps*p.duration())
        buffer_file = f'{ASSETS_ROOT}/mouth/{character_id}/assets_composited/mouth_0{suffix}.png'
        # buffer_file = f'./ARPABET_VISEMES/0.png'
        buffer = cv2.imread(buffer_file)
        buffer = cv2.resize(buffer,(1024,1024))
        viseme_id=''
        if p.mark!='':
            try:
                viseme_id = phoneme2viseme[p.mark[:2]]
                viseme_file = f'{ASSETS_ROOT}/mouth/{character_id}/assets_composited/mouth_{viseme_id}{suffix}.png'
                # viseme_file = f'./ARPABET_VISEMES/{viseme_id}.png'
                viseme_im = cv2.imread(viseme_file)
                viseme_im = cv2.resize(viseme_im,(1024,1024))
                buffer = viseme_im
            except:
                pass
        for _ in range(frame_count):
            if cumulative_frame_count>=word_frame_count:
                break
            # buffer = cv2.putText(buffer, f'{p.mark}->{viseme_id}', (500, 200), cv2.FONT_HERSHEY_SIMPLEX, 2, (0,255,0), 5)
            video.write(buffer)
            # cv2.imwrite(f'output/{character_id}/{i}_{cumulative_frame_count}_{p.mark}.png', buffer)
            cumulative_frame_count += 1
            
video.release() 

OpenCV: FFMPEG: tag 0x58564944/'DIVX' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
  0%|          | 0/209 [00:00<?, ?it/s]

100%|██████████| 209/209 [01:00<00:00,  3.48it/s]


In [15]:
ffmpeg_path = '/mnt/users_scratch/astitva/WORKSPACE/ffmpeg/installation/bin'
command = f'{ffmpeg_path}/ffmpeg -i ./tmp.mp4 -i {ALIGNER_ROOT}/inputs/{filename}/{filename}.wav -map 0:v:0 -map 1:a:0 -c:v copy -framerate {30}/1 ./lipsync_{filename}_{character_id}.mp4'
os.system(f'rm lipsync_{filename}_{character_id}.mp4')
os.system(command)

rm: cannot remove 'lipsync_babyshark_07aedcb335a04981a016c0c7efed77ba.mp4': No such file or directory
ffmpeg version 5.0 Copyright (c) 2000-2022 the FFmpeg developers
  built with gcc 11 (GCC)
  configuration: --prefix=/mnt/users_scratch/astitva/WORKSPACE/ffmpeg/installation --disable-debug --disable-x86asm
  libavutil      57. 17.100 / 57. 17.100
  libavcodec     59. 18.100 / 59. 18.100
  libavformat    59. 16.100 / 59. 16.100
  libavdevice    59.  4.100 / 59.  4.100
  libavfilter     8. 24.100 /  8. 24.100
  libswscale      6.  4.100 /  6.  4.100
  libswresample   4.  3.100 /  4.  3.100
Input #0, mov,mp4,m4a,3gp,3g2,mj2, from './tmp.mp4':
  Metadata:
    major_brand     : isom
    minor_version   : 512
    compatible_brands: isomiso2mp41
    encoder         : Lavf59.27.100
  Duration: 00:00:51.26, start: 0.000000, bitrate: 5946 kb/s
  Stream #0:0[0x1](und): Video: mpeg4 (Simple Profile) (mp4v / 0x7634706D), yuv420p, 1024x1024 [SAR 1:1 DAR 1:1], 5938 kb/s, 240 fps, 240 tbr, 15360 tbn 

0

In [ ]:
# from IPython.display import Video
# # Video("tmp.mp4")

### MOUTH + EYES (BABYSHARK)

In [ ]:
def composite(base, mouth, eyes, use_default_mouth=False, use_default_eyes=False):
    mask_mouth = mouth[:,:,3]/255
    mask_eyes = eyes[:,:,3]/255
    mask = mask_mouth + mask_eyes
    mask_im = np.repeat(mask[..., np.newaxis], 3, axis=2)
    mask_mouth_im = np.repeat(mask_mouth[..., np.newaxis], 3, axis=2)
    mask_eyes_im = np.repeat(mask_eyes[..., np.newaxis], 3, axis=2)
    composited = base*(1-mask_im) 
    if not use_default_eyes:
        composited += mask_eyes_im*eyes[:,:,:3]
    if not use_default_mouth:
        composited += mask_mouth_im*mouth[:,:,:3]
    return composited.astype('uint8')

In [ ]:
# character_id = '0a3b9f4c787743458c7ca1cc77b902ea'
# BASE_ROOT = './generated_assets/base/' + character_id
# MOUTH_ROOT = './generated_assets/mouth/' + character_id
# EYES_ROOT = './generated_assets/eyes/' + character_id

# mouth_id = 3
# eyes_id = 0

# base_im = cv2.imread(f'{BASE_ROOT}/{character_id}_base_face.png')
# mouth_im = cv2.imread(f'{MOUTH_ROOT}/{character_id}_arpabets_{mouth_id}_asset.png',-1)
# eyes_im = cv2.imread(f'{EYES_ROOT}/{character_id}_eyes_{eyes_id}_asset.png', -1)
# import matplotlib.pyplot as plt
# im = composite(base_im, mouth_im, eyes_im)
# plt.imshow(im)
# plt.show()

In [ ]:
filename = 'babyshark'
tg_path = f'/home/astitva/WORKSPACE/ForcedAligner/outputs/{filename}.TextGrid'
tg = textgrid.TextGrid.fromFile(tg_path)

words = tg[0]
phonemes = tg[1]

words_phonemes=[]
phoneme_idx = 0
for w in words:
    current_set = []
    time = w.duration()
    start = 0
    while(time!=start):
        ph = phonemes[phoneme_idx]
        start += ph.duration()
        current_set.append(ph)
        phoneme_idx += 1
    words_phonemes.append(current_set)
    

In [ ]:
eye_word_mapping = {'mommy':3,'daddy':0,'grandma':6,'grandpa':7, 'hunt':5}

In [ ]:
character_id = '0a0be5b3db37407cb434c5e0dc3cf70b'

BASE_ROOT = './generated_assets/base/' + character_id
MOUTH_ROOT = './generated_assets/mouth/' + character_id
EYES_ROOT = './generated_assets/eyes/' + character_id

fps=240

base = cv2.imread(f'{BASE_ROOT}/{character_id}_no_mouth_face.png')
eyes_id = 3
blink_id = 2
blink_gap = 2 #seconds
num_blink_frames = 10

USE_DEFAULT_MOUTH = False
USE_DEFAULT_EYES = True


video=cv2.VideoWriter(f'tmp.mp4',cv2.VideoWriter_fourcc(*'DIVX'),fps,(1024,1024))
buffer = np.ones((1024,1024,3)).astype('uint8')*255
current_frame_count = 0

for i in tqdm(range(len(words))):
    total_duration = words[i].duration()
    word_frame_count = int(fps*total_duration)
    cumulative_frame_count = 0

    #eyes asset
    try:
        eyes_id = eye_word_mapping[words[i].mark]
        base = cv2.imread(f'{BASE_ROOT}/{character_id}_base_face.png')
        USE_DEFAULT_EYES=False
    except:
        pass

    eyes = cv2.imread(f'{EYES_ROOT}/{character_id}_eyes_{eyes_id}_asset.png', -1)
    
    for p in words_phonemes[i]:
        frame_count = math.ceil(fps*p.duration())

        #default mouth asset
        mouth_id = 0
        buffer_mouth = cv2.imread(f'{MOUTH_ROOT}/{character_id}_arpabets_{mouth_id}_asset.png',-1)
        
        # default
        buffer = composite(base, buffer_mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
        
        viseme_id=''
        if p.mark!='':
            try:
                mouth_id = phoneme2viseme[p.mark[:2]]
                mouth = cv2.imread(f'{MOUTH_ROOT}/{character_id}_arpabets_{mouth_id}_asset.png',-1)
                buffer = composite(base, mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
            except:
                pass
        for _ in range(frame_count):
            if cumulative_frame_count>=word_frame_count:
                break
            #blink
            if current_frame_count%(fps*blink_gap)<num_blink_frames:
                blink_eyes =  cv2.imread(f'{EYES_ROOT}/{character_id}_eyes_{blink_id}_asset.png', -1)
                blink_base = cv2.imread(f'{BASE_ROOT}/{character_id}_base_face.png')
                video.write(composite(blink_base, mouth, blink_eyes))
            else:
                video.write(buffer)
            cumulative_frame_count += 1
            current_frame_count += 1
            
            
video.release() 

In [ ]:
command = f'ffmpeg -i ./tmp.mp4 -i /home/astitva/WORKSPACE/ForcedAligner/inputs/{filename}/{filename}.wav -map 0:v:0 -map 1:a:0 -c:v copy -framerate {30}/1 ./animated_{filename}_{character_id}.mp4'
os.system(f'rm animated_{filename}_{character_id}.mp4')
os.system(command)

### MOUTH + EYES (SPEECH)

In [22]:
def composite(base, mouth, eyes, use_default_mouth=False, use_default_eyes=False):
    mask_mouth = mouth[:,:,3]/255
    mask_eyes = eyes[:,:,3]/255
    mask = mask_mouth + mask_eyes
    if use_default_eyes:
      mask = mask_mouth
    if use_default_mouth:
        mask = mask_eyes
    mask_im = np.repeat(mask[..., np.newaxis], 3, axis=2)
    mask_mouth_im = np.repeat(mask_mouth[..., np.newaxis], 3, axis=2)
    mask_eyes_im = np.repeat(mask_eyes[..., np.newaxis], 3, axis=2)
    composited = base*(1-mask_im) 
    if not use_default_eyes:
        composited += mask_eyes_im*eyes[:,:,:3]
    if not use_default_mouth:
        composited += mask_mouth_im*mouth[:,:,:3]
    return composited.astype('uint8')

In [85]:
filename = 'babyshark'
tg_path = f'/home/astitva/WORKSPACE/ForcedAligner/outputs/{filename}.TextGrid'
tg = textgrid.TextGrid.fromFile(tg_path)

words = tg[0]
phonemes = tg[1]

words_phonemes=[]
phoneme_idx = 0
for w in words:
    current_set = []
    time = w.duration()
    start = 0
    while(time!=start):
        ph = phonemes[phoneme_idx]
        start += ph.duration()
        current_set.append(ph)
        phoneme_idx += 1
    words_phonemes.append(current_set)
    

In [86]:
words

IntervalTier(words, [Interval(0.0, 0.88, None), Interval(0.88, 1.11, baby), Interval(1.11, 1.91, None), Interval(1.91, 2.04, shark), Interval(2.04, 2.41, doo), Interval(2.41, 2.58, None), Interval(2.58, 2.76, doo), Interval(2.76, 3.04, doo), Interval(3.04, 3.51, None), Interval(3.51, 4.01, doo), Interval(4.01, 4.16, None), Interval(4.16, 4.28, doo), Interval(4.28, 4.45, doo), Interval(4.45, 4.55, None), Interval(4.55, 4.99, baby), Interval(4.99, 5.09, None), Interval(5.09, 5.35, shark), Interval(5.35, 5.46, doo), Interval(5.46, 5.57, None), Interval(5.57, 5.81, doo), Interval(5.81, 5.85, None), Interval(5.85, 5.96, doo), Interval(5.96, 6.1, doo), Interval(6.1, 6.25, None), Interval(6.25, 6.36, doo), Interval(6.36, 6.54, doo), Interval(6.54, 6.64, None), Interval(6.64, 7.09, baby), Interval(7.09, 7.18, None), Interval(7.18, 7.43, shark), Interval(7.43, 7.53, doo), Interval(7.53, 7.66, None), Interval(7.66, 7.92, doo), Interval(7.92, 7.96, None), Interval(7.96, 8.07, doo), Interval(8.07,

In [87]:
# eye_word_mapping = {'please':2, 'thankyou':3}
eye_word_mapping = {'mommy':3,'daddy':0,'grandma':6,'grandpa':7, 'hunt':5}


In [91]:
# character_id = '07b97debed234daaa04313b000637b81'
characters = ['07b97debed234daaa04313b000637b81', '07ad6ccc1ac34288ab7c4f0a013ba3c3','07aedcb335a04981a016c0c7efed77ba', '07b1a55d68b9425caccb1aadcc58379a', '07b6c37d7a6944ee98f544d633defeeb', '07b8bf4a421744c9b7f985cf6e8fe544', '07c4c1c55b5b4098bf8f5b96defd6d2c']
# characters = ['0a3b9f4c787743458c7ca1cc77b902ea', '0a4a8a95ac934f4e8d7b58561f7913c9', '0a0be5b3db37407cb434c5e0dc3cf70b', '0a0cd1cc72f44d418e4884c7a402b030', '0a6bf1b9d15842b6822a92a6b536faf1', '0a5b805185614f839b5f015b650970dc']

skip_start_frames = 0
if filename=='babyshark':
    skip_start_frames = 1063

for character_id in characters:
    BASE_ROOT = './generated_assets/base/' + character_id
    MOUTH_ROOT = './generated_assets/mouth/' + character_id
    EYES_ROOT = './generated_assets/eyes/' + character_id
    
    fps=240
    
    base = cv2.imread(f'{BASE_ROOT}/{character_id}_no_mouth_face.png')
    eyes_id = 0
    blink_id = 2
    blink_gap = 2 #seconds
    num_blink_frames = 12
    
    USE_DEFAULT_MOUTH = False
    USE_DEFAULT_EYES = True
    
    
    video=cv2.VideoWriter(f'tmp.mp4',cv2.VideoWriter_fourcc(*'DIVX'),fps,(1024,1024))
    buffer = np.ones((1024,1024,3)).astype('uint8')*255
    current_frame_count = 0
    
    for i in tqdm(range(len(words))):
        total_duration = words[i].duration()
        word_frame_count = int(fps*total_duration)
        cumulative_frame_count = 0
    
        #eyes asset
        try:
            eyes_id = eye_word_mapping[words[i].mark]
            base = cv2.imread(f'{BASE_ROOT}/{character_id}_base_face.png')
            USE_DEFAULT_EYES=False
        except:
            pass
    
        eyes = cv2.imread(f'{EYES_ROOT}/{character_id}_eyes_{eyes_id}_asset.png', -1)
        
        for p in words_phonemes[i]:
            frame_count = math.ceil(fps*p.duration())
    
            #default mouth asset
            mouth_id = 0
            buffer_mouth = cv2.imread(f'{MOUTH_ROOT}/{character_id}_arpabets_{mouth_id}_asset.png',-1)
            
            # default
            buffer = composite(base, buffer_mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
            
            viseme_id=''
            if p.mark!='':
                try:
                    mouth_id = phoneme2viseme[p.mark[:2]]
                    mouth = cv2.imread(f'{MOUTH_ROOT}/{character_id}_arpabets_{mouth_id}_asset.png',-1)
                    buffer_mouth = mouth
                    buffer = composite(base, mouth, eyes, USE_DEFAULT_MOUTH, USE_DEFAULT_EYES)
                except:
                    pass
            # print(USE_DEFAULT_EYES)
            for _ in range(frame_count):
                if cumulative_frame_count>=word_frame_count:
                    break
                #blink
                if current_frame_count%(fps*blink_gap)<num_blink_frames:
                    blink_eyes =  cv2.imread(f'{EYES_ROOT}/{character_id}_eyes_{blink_id}_asset.png', -1)
                    blink_base = cv2.imread(f'{BASE_ROOT}/{character_id}_base_face.png')
                    if current_frame_count>skip_start_frames:
                        video.write(composite(blink_base, buffer_mouth, blink_eyes))
                else:
                    if current_frame_count>skip_start_frames:
                        video.write(buffer)
                cumulative_frame_count += 1
                current_frame_count += 1
                
                
    video.release() 

    command = f'ffmpeg -i ./tmp.mp4 -i /home/astitva/WORKSPACE/ForcedAligner/inputs/{filename}/{filename}_trimmed.wav -map 0:v:0 -map 1:a:0 -c:v copy -framerate {60}/1 ./animated_{filename}_{character_id}.mp4'
    os.system(f'rm animated_{filename}_{character_id}.mp4')
    os.system(command)

OpenCV: FFMPEG: tag 0x58564944/'DIVX' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
100%|█████████████████████████████████████████| 209/209 [02:08<00:00,  1.63it/s]
rm: cannot remove 'animated_babyshark_07b97debed234daaa04313b000637b81.mp4': No such file or directory
ffmpeg version 6.1.2 Copyright (c) 2000-2024 the FFmpeg developers
  built with gcc 14 (GCC)
  configuration: --prefix=/usr --bindir=/usr/bin --datadir=/usr/share/ffmpeg --docdir=/usr/share/doc/ffmpeg --incdir=/usr/include/ffmpeg --libdir=/usr/lib64 --mandir=/usr/share/man --arch=x86_64 --optflags='-O2 -flto=auto -ffat-lto-objects -fexceptions -g -grecord-gcc-switches -pipe -Wall -Werror=format-security -Wp,-U_FORTIFY_SOURCE,-D_FORTIFY_SOURCE=3 -Wp,-D_GLIBCXX_ASSERTIONS -specs=/usr/lib/rpm/redhat/redhat-hardened-cc1 -fstack-protector-strong -specs=/usr/lib/rpm/redhat/redhat-annobin-cc1 -m64 -march=x86-64 -mtune=generic -fasynchronous-unwind-t

In [ ]:
# command = f'ffmpeg -i ./tmp.mp4 -i /home/astitva/WORKSPACE/ForcedAligner/inputs/{filename}/{filename}.wav -map 0:v:0 -map 1:a:0 -c:v copy -framerate {60}/1 ./animated_{filename}_{character_id}.mp4'
# os.system(f'rm animated_{filename}_{character_id}.mp4')
# os.system(command)